In [1]:
import os

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfTransformer

In [2]:
data = pd.read_csv("../data/snsdata.csv")
data.head()

,gradyear,gender,age,friends,basketball,football,soccer,softball,volleyball,swimming,...,blonde,mall,shopping,clothes,hollister,abercrombie,die,death,drunk,drugs
0,2006,M,18.982,7,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2006,F,18.801,0,0,1,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
2,2006,M,18.335,69,0,1,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,2006,F,18.875,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2006,NaN,18.995,10,0,0,0,0,0,0,...,0,0,2,0,0,0,0,0,1,1


In [3]:
# Variables de caracterización:
# - gradyear: año de graduación;
# - gender: género;
# - age: edad;
# - friends`: número de amigos.

data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 40 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   gradyear      30000 non-null  int64  
 1   gender        27276 non-null  object 
 2   age           24914 non-null  float64
 3   friends       30000 non-null  int64  
 4   basketball    30000 non-null  int64  
 5   football      30000 non-null  int64  
 6   soccer        30000 non-null  int64  
 7   softball      30000 non-null  int64  
 8   volleyball    30000 non-null  int64  
 9   swimming      30000 non-null  int64  
 10  cheerleading  30000 non-null  int64  
 11  baseball      30000 non-null  int64  
 12  tennis        30000 non-null  int64  
 13  sports        30000 non-null  int64  
 14  cute          30000 non-null  int64  
 15  sex           30000 non-null  int64  
 16  sexy          30000 non-null  int64  
 17  hot           30000 non-null  int64  
 18  kissed        30000 non-nu

In [4]:
# Calidad de datos:

data[["gradyear", "gender", "age", "friends"]].describe(include="all")




,gradyear,gender,age,friends
count,30000.000000,27276,24914.000000,30000.000000
unique,NaN,2,NaN,NaN
top,NaN,F,NaN,NaN
freq,NaN,22054,NaN,NaN
mean,2007.500000,NaN,17.993950,30.179467
std,1.118053,NaN,7.858054,36.530877
min,2006.000000,NaN,3.086000,0.000000
25%,2006.750000,NaN,16.312000,3.000000
50%,2007.500000,NaN,17.287000,20.000000
75%,2008.250000,NaN,18.259000,44.000000


In [5]:

# Diagnóstico de la variable edad:

print("Edades faltantes:", data["age"].isna().sum())
print("Edad mínima:", data["age"].min())
print("Edad máxima:", data["age"].max())



Edades faltantes: 5086
Edad mínima: 3.086
Edad máxima: 106.927


In [6]:
# Remoción de datos con edades fuera del rango de 13 a 20 años (inclusive):

data["age_clean"] = data["age"].where(data["age"].between(13, 20, inclusive="left"))

age_by_gradyear = (
    data.groupby("gradyear")["age_clean"].agg(["count", "mean", "median"]).round(3)
)

age_by_gradyear



,count,mean,median
gradyear,,,
2006,6188,18.656,18.672
2007,6219,17.706,17.689
2008,6105,16.768,16.734
2009,5965,15.820,15.786


In [7]:

# Imputación de edades faltantes con la mediana de edad por año de graduación:

age_medians = data.groupby("gradyear")["age_clean"].median()

data["age_imputed"] = data["age_clean"].fillna(data["gradyear"].map(age_medians))

print("Edades faltantes antes de imputar:", data["age_clean"].isna().sum())
print("Edades faltantes después de imputar:", data["age_imputed"].isna().sum())

data[["gradyear", "age", "age_clean", "age_imputed"]].head(10)

	





Edades faltantes antes de imputar: 5523
Edades faltantes después de imputar: 0


,gradyear,age,age_clean,age_imputed
0,2006,18.982,18.982,18.982
1,2006,18.801,18.801,18.801
2,2006,18.335,18.335,18.335
3,2006,18.875,18.875,18.875
4,2006,18.995,18.995,18.995
5,2006,NaN,NaN,18.672
6,2006,18.930,18.930,18.930
7,2006,18.322,18.322,18.322
8,2006,19.055,19.055,19.055
9,2006,18.708,18.708,18.708


In [8]:
# Selección de las variables de intereses (columnas `basketball` a `drugs`):

interests = data.loc[:, "basketball":"drugs"].columns.tolist()

print("Número de variables de intereses:", len(interests))
print("Usuarios sin ninguna palabra de interés:", (data[interests].sum(axis=1) == 0).sum())

data[interests].describe().T.head()

Número de variables de intereses: 36
Usuarios sin ninguna palabra de interés: 2473


,count,mean,std,min,25%,50%,75%,max
basketball,30000.0,0.267333,0.804708,0.0,0.0,0.0,0.0,24.0
football,30000.0,0.252300,0.705357,0.0,0.0,0.0,0.0,15.0
soccer,30000.0,0.222767,0.917226,0.0,0.0,0.0,0.0,27.0
softball,30000.0,0.161200,0.739707,0.0,0.0,0.0,0.0,17.0
volleyball,30000.0,0.143133,0.639943,0.0,0.0,0.0,0.0,14.0


In [9]:
# Transformación TF-IDF de las frecuencias de palabras.
#
# Se pondera cada término por su rareza en la población (IDF) y se normaliza cada usuario para
# que la comparación dependa del perfil de intereses y no de cuántas palabras escribió.

tfidf = TfidfTransformer(sublinear_tf=True)
X = tfidf.fit_transform(data[interests])

X.shape

(30000, 36)

In [10]:
# Agrupamiento con k-means (5 segmentos):

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
data["cluster"] = kmeans.fit_predict(X)

data["cluster"].value_counts().sort_index()

cluster
0    14864
1     3907
2     3654
3     4885
4     2690
Name: count, dtype: int64

In [11]:
# Perfil de los clusters: tamaño, edad y amigos, y términos más distintivos de cada centroide.

cluster_profile = data.groupby("cluster").agg(
    n=("cluster", "size"),
    age_mean=("age_imputed", "mean"),
    friends_median=("friends", "median"),
)
cluster_profile["age_mean"] = cluster_profile["age_mean"].round(3)

centroids = pd.DataFrame(kmeans.cluster_centers_, columns=interests)
centroids_diff = centroids - centroids.mean()

fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharex=True)

for cluster, ax in zip(centroids_diff.index, axes):
    top_terms = centroids_diff.loc[cluster].sort_values().tail(8)
    top_terms.plot.barh(ax=ax)
    ax.set_xlabel(f"Cluster {cluster}")

plt.tight_layout()
plt.show()

cluster_profile

<Figure size 2000x400 with 5 Axes>

,n,age_mean,friends_median
cluster,,,
0,14864,17.273,18.0
1,3907,17.360,18.0
2,3654,17.111,24.0
3,4885,17.070,22.0
4,2690,17.302,22.0


In [12]:
# Interpretación de los clusters
#
# En la ejecución con `random_state=42`, los perfiles dominantes son aproximadamente:
#
#  Cluster 0 – Deportes, religión y baja intensidad de intereses específicos
#    Es el cluster más grande y sus centroides están cerca del promedio en casi todas las variables.
#    Se distinguen levemente `basketball`, `soccer`, `volleyball`, `softball`, `football`, `church`
#    y `jesus`.
#
#    Interpretación: usuarios con menor intensidad de señal en las categorías analizadas o con
#    intereses poco especializados, con cierta inclinación hacia deportes y religión.
#
#  Cluster 1 – Música y rock
#    Sobresalen `music`, `rock` y, en menor medida, `shopping`.
#
#    Interpretación: segmento orientado al consumo de música.
#
#  Cluster 2 – Baile, moda y apariencia
#    Destacan `dance`, `cute`, `shopping`, `dress`, `cheerleading`, `hot` y `mall`.
#
#    Interpretación: segmento orientado a baile, porrismo, moda y compras.
#
#  Cluster 3 – Vida social y comportamientos de riesgo
#    Los términos más distintivos incluyen
#                    `hair`, `kissed`, `sex`, `blonde`, `clothes`, `cute`, `drugs` y `mall`.
#
#    Interpretación: perfiles con conversación más intensa alrededor de relaciones, apariencia,
#    vida social y comportamientos de riesgo.
#
#  Cluster 4 – Banda y música escolar
#    Los términos más distintivos son
#                                `band` y `marching`.
#
#    Interpretación: nicho pequeño y muy especializado alrededor de bandas escolares y
#    actividades musicales.
#
# NOTA: Los nombres de los segmentos son **etiquetas analíticas construidas después del clustering.
# El algoritmo no conoce ni genera estos nombres: estos surgen de la interpretación de los perfiles.

In [13]:
# Tabla resumen para una decisión de marketing

segment_names = {
    0: "Deportes, religión y baja intensidad de intereses específicos",
    1: "Música y rock",
    2: "Baile, moda y apariencia",
    3: "Vida social y comportamientos de riesgo",
    4: "Banda y música escolar",
}

summary = cluster_profile.copy()
summary["segment"] = summary.index.map(segment_names)

summary = summary[["segment", "n", "age_mean", "friends_median"]]

summary

,segment,n,age_mean,friends_median
cluster,,,,
0,"Deportes, religión y baja intensidad de intere...",14864,17.273,18.0
1,Música y rock,3907,17.360,18.0
2,"Baile, moda y apariencia",3654,17.111,24.0
3,Vida social y comportamientos de riesgo,4885,17.070,22.0
4,Banda y música escolar,2690,17.302,22.0


In [14]:
#  Almacena el resultado

output = data.copy()
output["segment"] = output["cluster"].map(segment_names)

output.to_csv(
    "../submission/segmented.csv",
    index=False,
)
